Small Example to test correctness with N=4, M=2 and an anisotropic number of blocks.

In [162]:
%%writefile mat_calc.cu

#include "cuda_runtime.h"
#include <stdlib.h>
#include <stdio.h>
#include <iostream>

#define N 4
#define M 2
#define THREADS 4

__global__ void matmult(float *a, float *b, float *x, float *c, int n, int m) {
  int row = threadIdx.x + blockIdx.x * blockDim.x;
  int column = threadIdx.y + blockIdx.y * blockDim.y;

  int thread_id = row * m + column;
  while (thread_id < n*n) {
    int i = thread_id / n;
    float sum = 0;
    for (int j= 0; j<m; j++) {
      sum += a[i * m + j] * x[i + j * n];
    }

    c[thread_id] = sum + b[i];
    thread_id += blockDim.x * gridDim.x;
  }
}

void fill_a(float *a){
  for(int i=0; i<N; i++) {
    for(int j=0; j<M; j++) {
      a[j+i*M] = i < N/2 ;
    }
  }

}

void fill_b(float *b){
  for(int i=0;i<N*N;i++) {
    b[i] = 0;
  }
}

void fill_x(float *x){
  for(int i=0;i<M;i++) {
    for(int j=0;j<N;j++) {
      x[j+i*N] = i<M/2;
    }
  }
}

int main() {
  float *a, *b, *c, *x;
  float *d_a, *d_b, *d_c, *d_x;

  const int mat_size = N*M*sizeof(float);
  const int output_size = N*N*sizeof(float);

  cudaMalloc((void**)&d_a, mat_size);
	cudaMalloc((void**)&d_b, output_size);
	cudaMalloc((void**)&d_c, output_size);
  cudaMalloc((void**)&d_x, mat_size);

  a = (float*)malloc(mat_size);
	b = (float*)malloc(output_size);
	c = (float*)malloc(output_size);
  x = (float*)malloc(mat_size);

  fill_a(a);
  fill_b(b);
  fill_x(x);

  cudaMemcpy(d_a, a, mat_size, cudaMemcpyHostToDevice);
  cudaMemcpy(d_b, b, output_size, cudaMemcpyHostToDevice);
  cudaMemcpy(d_x, x, mat_size, cudaMemcpyHostToDevice);

  dim3 blocks(2, 4);
  dim3 threads(2, 2);

  matmult << <blocks, threads >> >(d_a, d_b, d_x, d_c, N, M);

  cudaMemcpy(c, d_c, output_size, cudaMemcpyDeviceToHost);

  std::cout << "A: " << std::endl;
  for(int i=0; i<N; i++) {
    for(int j=0; j<M; j++){
      std::cout << a[i*M + j] << " ";
    }
    std::cout << std::endl;
  }

  std::cout << "X: " << std::endl;
  for(int i=0; i<M; i++) {
    for(int j=0; j<N; j++){
      std::cout << x[i*N + j] << " ";
    }
    std::cout << std::endl;
  }

  std::cout << "B: " << std::endl;
  for(int i=0; i<N; i++) {
    for(int j=0; j<N; j++){
      std::cout << b[i*N + j] << " ";
    }
    std::cout << std::endl;
  }


  std::cout << "C: " << std::endl;
  for(int i=0; i<N; i++) {
    for(int j=0; j<N; j++){
      std::cout << c[i*N + j] << " ";
    }
    std::cout << std::endl;
  }

  free(a);
  free(b);
  free(x);
  free(c);

  cudaFree(a);
  cudaFree(b);
  cudaFree(x);
  cudaFree(c);

  return 0;

}



Overwriting mat_calc.cu


In [163]:
!nvcc -arch=sm_75 mat_calc.cu -o mat_calc
!./mat_calc

A: 
1 1 
1 1 
0 0 
0 0 
X: 
1 1 1 1 
0 0 0 0 
B: 
0 0 0 0 
0 0 0 0 
0 0 0 0 
0 0 0 0 
C: 
1 1 1 1 
1 1 1 1 
0 0 0 0 
0 0 0 0 


Example to measure time between single and multi core

In [171]:
%%writefile mat_calc_timed.cu

#include "cuda_runtime.h"
#include <stdlib.h>
#include <stdio.h>
#include <iostream>
#include <chrono>

#define N 8192 /  2
#define M 2048
#define THREADS 4

__global__ void matmult(float *a, float *b, float *x, float *c, int n, int m) {
  int row = threadIdx.x + blockIdx.x * blockDim.x;
  int column = threadIdx.y + blockIdx.y * blockDim.y;

  int thread_id = row * m + column;
  while (thread_id < n*n) {
    int i = thread_id / n;
    float sum = 0;
    for (int j= 0; j<m; j++) {
      sum += a[i * m + j] * x[i + j * n];
    }

    c[thread_id] = sum + b[i];
    thread_id += blockDim.x * gridDim.x;
  }
}

void fill_a(float *a){
  for(int i=0; i<N; i++) {
    for(int j=0; j<M; j++) {
      a[j+i*M] = i < N/2 ;
    }
  }

}

void fill_b(float *b){
  for(int i=0;i<N*N;i++) {
    b[i] = 0;
  }
}

void fill_x(float *x){
  for(int i=0;i<M;i++) {
    for(int j=0;j<N;j++) {
      x[j+i*N] = i<M/2;
    }
  }
}

int main() {
  float *a, *b, *c, *x;
  float *d_a, *d_b, *d_c, *d_x;

  const int mat_size = N*M*sizeof(float);
  const int output_size = N*N*sizeof(float);

  cudaMalloc((void**)&d_a, mat_size);
	cudaMalloc((void**)&d_b, output_size);
	cudaMalloc((void**)&d_c, output_size);
  cudaMalloc((void**)&d_x, mat_size);

  a = (float*)malloc(mat_size);
	b = (float*)malloc(output_size);
	c = (float*)malloc(output_size);
  x = (float*)malloc(mat_size);

  fill_a(a);
  fill_b(b);
  fill_x(x);

  cudaMemcpy(d_a, a, mat_size, cudaMemcpyHostToDevice);
  cudaMemcpy(d_b, b, output_size, cudaMemcpyHostToDevice);
  cudaMemcpy(d_x, x, mat_size, cudaMemcpyHostToDevice);

  // Single Core
  dim3 blocks_single(1, 1);
  dim3 threads_single(1, 1);
  auto start_single = std::chrono::high_resolution_clock::now();
  matmult << <blocks_single, threads_single >> >(d_a, d_b, d_x, d_c, N, M);
  auto stop_single = std::chrono::high_resolution_clock::now();

  // Multi Core
  dim3 blocks_multi((N / THREADS) / 2, (N / THREADS) / 2);
  dim3 threads_multi(THREADS, THREADS);
  auto start_multi = std::chrono::high_resolution_clock::now();
  matmult << <blocks_multi, threads_multi >> >(d_a, d_b, d_x, d_c, N, M);
  auto stop_multi = std::chrono::high_resolution_clock::now();

  cudaMemcpy(c, d_c, output_size, cudaMemcpyDeviceToHost);

  free(a);
  free(b);
  free(x);
  free(c);

  cudaFree(a);
  cudaFree(b);
  cudaFree(x);
  cudaFree(c);

  auto single_time = std::chrono::duration_cast< std::chrono::microseconds>(stop_single - start_single).count();
  auto multi_time = std::chrono::duration_cast< std::chrono::microseconds>(stop_multi - start_multi).count();

  std::cout << "Single Core: " <<  single_time << std::endl;
  std::cout << "Multi Core: " << multi_time << std::endl;
  std::cout << "Speedup: " << single_time / multi_time << std::endl;

  return 0;

}



Overwriting mat_calc_timed.cu


In [ ]:
!nvcc -arch=sm_75 mat_calc_timed.cu -o mat_calc_timed
!./mat_calc_timed